## Built a Simple LLM Application with LCEL 
Project Name -- LanguageTranslatorGroq&OpenAI

In this quickstart, we'll build a simple LLM application using LanChain. This application will translate text from English language into another language. This is relatively simple LLM application, it's just a single LLM call with some prompting. Still, this is a great way to start with LangChain. A lot of features can be built with just some prompting and LLM call.

After this project, we'll have a high level overview of:

- Using language models
- Using prompt template and output parsers
- Using LangChain Expression Language to chain components together
- Debugging and Tracing your application with LangSmith
- Deploying your application with LangServe

#### What is Groq?

Groq is a company that provides very fast cloud inference for AI models.

Think of it like:

OpenAI
   ↓
Provides GPT models via API

Groq
   ↓
Provides open-source models via API

Example models available on Groq:

Llama 3
Gemma
DeepSeek-R1
Qwen

You call them through an API just like OpenAI:

In [2]:
### OpenAI API (Paid) and the Groq API (Free)

import os
from dotenv import load_dotenv

load_dotenv(override=True)

## For OpenAI Models API
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

## For Groq API
groq_api_key = os.getenv("GROQ_API_KEY")

## For LangSmith Tracking
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT") or os.getenv("LANGCHAIN_PROJECT")
 
# Usually optional for hosted LangSmith, but safe to set
os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")

In [3]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x1278f4070>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1278f40d0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content = "Translate the following from English to French"),
    HumanMessage(content = "Hello, How are you?")
]

result = model.invoke(messages)

In [6]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
parser.invoke(result)

'Bonjour, comment allez-vous ?'

In [7]:
## Using LCEL to chain all the components like the model and the parser
chain = model | parser
chain.invoke(messages)

'Bonjour, Comment allez-vous ?'

In [10]:
## Now instead of passing the messages here directly we can use the PromptTemplates, which can be added to our chain
from langchain_core.prompts import ChatPromptTemplate

generic_template = "Convert the following into {language}"
prompt = ChatPromptTemplate.from_messages([
    ("system", generic_template),
    ("user", "{text}")
])

In [12]:
result = prompt.invoke({"language": "French", "text": "Hello"})

In [13]:
result.to_messages()

[SystemMessage(content='Convert the following into French', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [14]:
## Now chaining prompt, llm and the output parser
chain = prompt | model | parser
chain.invoke({"language": "French", "text": "Hello"})

'Bonjour'